### Import Dependencies

In [1]:
import yaml
from jinja2 import Template
from langsmith import Client

### RAG Pipeline Prompt

In [2]:
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
{preprocessed_context}

Question:
{question}
"""

    return prompt

In [3]:
preprocessed_context = "- a \n- b"
question = "What is a?"

In [4]:
prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
{preprocessed_context}

Question:
{question}
"""

In [5]:
print(prompt)


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
- a 
- b

Question:
What is a?



### Jinja Templates

In [6]:
jinja_template = """
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
{{ preprocessed_context }}

Question:
{{ question }}
"""

In [7]:
template = Template(jinja_template)

In [8]:
rendered_template = template.render(preprocessed_context=preprocessed_context, question=question)

In [9]:
print(rendered_template)


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
- a 
- b

Question:
What is a?


In [10]:
def build_prompt_jinja(preprocessed_context, question):

    jinja_template = """
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
{{ preprocessed_context }}

Question:
{{ question }}
"""

    template = Template(jinja_template)
    rendered_prompt = template.render(
        preprocessed_context=preprocessed_context,
        question=question
    )

    return rendered_prompt

In [11]:
print(build_prompt_jinja(preprocessed_context, question))


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
- a 
- b

Question:
What is a?


In [12]:
print(build_prompt_jinja("- some item", "My silly question."))


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
- some item

Question:
My silly question.


In [13]:
def prompt_template_config(yaml_path, prompt_key):

    with open(yaml_path, "r") as file:
        config = yaml.safe_load(file)

    template_content = config["prompts"][prompt_key]

    template = Template(template_content)

    return template

In [14]:
def build_prompt_jinja(preprocessed_context, question):

    template = prompt_template_config("prompts/retrieval_generation.yaml", "retrieval_generation")

    rendered_prompt = template.render(
        preprocessed_context=preprocessed_context,
        question=question
    )

    return rendered_prompt

In [15]:
print(build_prompt_jinja(preprocessed_context, question))


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
- a 
- b

Question:
What is a?


### Prompt Registries

In [16]:
ls_client = Client()

In [17]:
ls_template = ls_client.pull_prompt("retrieval-generation")

In [18]:
ls_template

ChatPromptTemplate(input_variables=[], input_types={}, partial_variables={}, metadata={'lc_hub_owner': '-', 'lc_hub_repo': 'retrieval-generation', 'lc_hub_commit_hash': '120a1e9d0cc514b722d1ebde887aa1d93e1f0d72928c8228f150c6f6b562912a'}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a shopping assistant that can answer questions about the products in stock.\n\nYou will be given a question and a list of context.\n\nInstructions:\n- Answer the question based on the provided context only.\n- Never use word context and refer to it as the available products.\n\nContext:\n{{ preprocessed_context }}\n\nQuestion:\n{{ question }}'), additional_kwargs={})])

In [19]:
print(ls_template.messages[0].prompt.template)

You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
{{ preprocessed_context }}

Question:
{{ question }}


In [20]:
def prompt_template_registry(prompt_name):

    template_content = ls_client.pull_prompt(prompt_name).messages[0].prompt.template

    template = Template(template_content)

    return template

In [21]:
print(prompt_template_registry("retrieval-generation").render(
        preprocessed_context=preprocessed_context,
        question=question
    ))

You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
- a 
- b

Question:
What is a?
